# Finding good comparison stars for our target stars
The following code  download photometry from the SkyMapper survey around a specificed RA and DEC and search for good comparison stars for potential transiting exoplanet target stars. .
<br>
After the code has found the comparison stars, it  outputs the number of comparison stars for Mookodi and SHOC.
<br> 
<br>
General information on Skymapper: [https://skymapper.anu.edu.au/about-skymapper/](https://skymapper.anu.edu.au/about-skymapper/)  . This page gives information on the meaning of the various columns: [https://skymapper.anu.edu.au/table-browser/](https://skymapper.anu.edu.au/table-browser/) <br>
<br>
To access the date, we will use TAP (Table Access Protocol), which is a table query based upon standards from the Virtual Observatory.  To access the TAP, we use the pyvo and regions package.  These are not installed by default, so you need to open up a terminal window and type in the following at the command line: <br>
pip install pyvo <br>
pip install regions
<br> 
<br>
The code below creates the TAP query, doing a cone search with a radius of 5.0 arcminutes = 0.083 degrees around the specified RA and DEC of the SkyMapper photometry file.  It specfies which columns to download, and the output is ordered by the distance of the star from the center of the search cone.



In [3]:
import pyvo as vo
import pandas as pd
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from regions import CircleSkyRegion
from regions import Regions
from io import BytesIO
import math 
import requests
from astropy.io import fits
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.patches import Rectangle
import matplotlib.text as mpl_text

mookodilist = []
shoclist = []

#dfin = pd.read_csv('TESS_TOI_possible.csv')
dfin = pd.read_csv('TESS_TOI_short.csv')

#print (dfin)

for row in dfin.itertuples():
    inra = row.RA
    indec = row.Dec
    my_gaia_id = row.TOI
    Gaia_G = row.TESS_Mag
#    print(inra,indec,my_gaia_id,Gaia_G)

    
    c = SkyCoord(inra,indec, unit=(u.hourangle, u.deg))
    myra=str(c.ra.degree)
    mydec=str(c.dec.degree)
    print(myra,mydec)
    
    
 #create the TAP query. Note that the search radius is in degrees, and 0.083 is 5.0 arcminutes
    my_tap_query = ("SELECT DISTANCE(POINT('ICRS', raj2000, dej2000), POINT('ICRS', "   
              + myra + ", " + mydec + 
              ")) AS dist, "  + 
              "m.object_id,m.raj2000,m.dej2000,m.g_flags,m.g_nimaflags,m.i_flags,m.i_nimaflags, " + 
            "m.g_psf,m.e_g_psf,m.i_psf,m.e_i_psf  FROM dr4.master AS m " + 
              "WHERE 1 = CONTAINS(POINT('ICRS', raj2000, dej2000), " 
              "CIRCLE('ICRS',"  + 
               myra +", "+ mydec + 
              ", 0.083 )) ORDER BY dist " )
              
#    print(my_tap_query)
   
# exectue the search on the SkyMapper server
    tap_service = vo.dal.TAPService("https://api.skymapper.nci.org.au/public/tap/")
    tap_results = tap_service.search(my_tap_query)

    # convert to a pandas datafram
    astropy_table = tap_results.to_table()
    df = astropy_table.to_pandas()
    print('df.shape:', df.shape)

    correctstar = True
#make sure that we really have found out target star 
    if df['dist'].iloc[0] > 0.00056 :
        correctstar = False

    if (correctstar) :
# get rid of stars with bad photometry        
        g_good1 = df['g_flags'] == 0 
        g_good2 = df['g_nimaflags'] == 0
        g_good3 = df['e_g_psf'] < 0.07
        i_good1 = df['i_flags'] == 0 
        i_good2 = df['i_nimaflags'] == 0
        i_good3 = df['e_i_psf'] < 0.07

#combine these comparisons to make a new dataframe
# using df_good = df.query('g_flags == 0 ') does not work, as some of the values are null...
        df_good = df[g_good1 & g_good2 & g_good3 & i_good1 & i_good2 & i_good3]
        num_good = df_good.shape[0]
#    print("Number of stars with good photometry: ",  num_good)
#    print(df_good[['dist', 'raj2000', 'dej2000','g_psf','e_g_psf', 'i_psf', 'e_i_psf']])

        i_brightcut = Gaia_G - 1.0
        i_faintcut = Gaia_G + 4.5

        i_bright = df_good['i_psf'] > i_brightcut
        i_faint = df_good['i_psf'] < i_faintcut

        df_mookodi = df_good[i_bright & i_faint]

        shoc_comp = df_mookodi['dist'] < 0.0233   #SHOC field of view is 2.8x2.8 arcminutes, so use r=1.4 arcminutes
        df_shoc = df_mookodi[shoc_comp]
        num_mookodi = df_mookodi.shape[0]
        num_shoc_comp = df_shoc.shape[0]
    else :
        num_mookodi = 0
        num_shoc_comp = 0
        
        
    print('Number Mookodi:', num_mookodi)
    mookodilist.append(num_mookodi)
    shoclist.append(num_shoc_comp)
#    print("mookodoi:", mookodilist)
#    print("shoc:",shoclist)

    
print (dfin.shape)
print (len(mookodilist))
print (len(shoclist))
dfin['Num_Mookodi'] = mookodilist
dfin['Num_SHOC'] = shoclist

# saving the DataFrame as a CSV file 
tess_csv_data = dfin.to_csv('TESS_comp_stars.csv', index = False) 
print('\nCSV String:\n', tess_csv_data) 
    

318.73699999999997 -55.87186388888889
df.shape: (199, 12)
Number Mookodi: 20
312.4575 -24.428694444444446
df.shape: (261, 12)
Number Mookodi: 9
313.78324999999995 -34.13575
df.shape: (343, 12)
Number Mookodi: 0
316.96149999999994 -26.096719444444442
df.shape: (188, 12)
Number Mookodi: 26
313.2154583333333 -25.687375
df.shape: (262, 12)
Number Mookodi: 54
5.618541666666665 -59.942588888888885
df.shape: (129, 12)
Number Mookodi: 6
332.7143333333333 -30.749719444444445
df.shape: (233, 12)
Number Mookodi: 7
63.37470833333333 -69.22659166666666
df.shape: (254, 12)
Number Mookodi: 4
6.702624999999999 -56.31610277777777
df.shape: (142, 12)
Number Mookodi: 17
(9, 62)
9
9

CSV String:
 None
